In [22]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
import os
import matplotlib.colors as mcolors

In [23]:
# folder_path='/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_division/'
folder_path='/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_div_7jul2025/'

os.chdir(folder_path)

In [24]:
# Loading the graphs
# fast_first_div=nx.read_graphml('/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_division/fast_first_div/network_dir/1_0.graphml')
# synchronous_div=nx.read_graphml('/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_division/synchronized_strain/network_dir/1_0.graphml')

fast_first_div=nx.read_graphml('/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_div_7jul2025/no_frag_growth_15_var_7july2025/test_15_var_-30_diff/network_dir/network_sim1_step199_n200.graphml')
synchronous_div=nx.read_graphml('/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_div_7jul2025/no_frag_growth_15_var_7july2025/test_15_var_0_diff/network_dir/network_sim1_step199_n200.graphml')



In [25]:
print(f"fast first divisions has {fast_first_div.number_of_nodes()} nodes.")
print(f"synchronized divisions has {synchronous_div.number_of_nodes()} nodes.")


fast first divisions has 200 nodes.
synchronized divisions has 200 nodes.


In [26]:

# Calculate node degrees for both networks
fast_first_div_degrees = dict(fast_first_div.degree())
synchronous_div_degrees = dict(synchronous_div.degree())


# Function to identify filament nodes
def identify_filament_nodes(graph, min_length=3):
    filament_nodes = set()
    visited = set()

    def traverse_filament(node):
        path = [node]
        current = node
        visited.add(current)

        while True:
            neighbors = list(graph.neighbors(current))
            unvisited_neighbors = [n for n in neighbors if n not in visited]

            if len(unvisited_neighbors) == 1 and graph.degree(unvisited_neighbors[0]) <= 2:
                next_node = unvisited_neighbors[0]
                path.append(next_node)
                visited.add(next_node)
                current = next_node
            else:
                break

        return path if len(path) >= min_length else []

    for node, degree in dict(graph.degree()).items():
        if degree == 1 and node not in visited:
            filament = traverse_filament(node)
            filament_nodes.update(filament)

    return filament_nodes

# Function to create and save network visualization
def create_network_visualization(graph, degrees, filename, node_size, edge_width, min_filament_length=3):
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    
    # Identify filament nodes
    filament_nodes = identify_filament_nodes(graph, min_filament_length)
    
    for node in nt.nodes:
        node_id = node['id']
        node['size'] = node_size
        node['label'] = ''
        if node_id in filament_nodes:
            node['color'] = 'red'
        else:
            node['color'] = 'darkgray'
    
    for edge in nt.edges:
        if edge['from'] in filament_nodes and edge['to'] in filament_nodes:
            edge['color'] = 'red'
        else:
            edge['color'] = 'darkgray'
        edge['width'] = edge_width
    
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

# Create visualizations for both networks
min_fil_size=3
# create_network_visualization(fast_first_div, fast_first_div_degrees, 'fast_first_div_filament_size_'+str(min_fil_size)+'.html', node_size=36, edge_width=18)
# create_network_visualization(synchronous_div, synchronous_div_degrees, 'synchronous_div_filament_size_'+str(min_fil_size)+'.html', node_size=36, edge_width=18)

create_network_visualization(fast_first_div, fast_first_div_degrees, 'fast_first_div_filament_size_'+str(min_fil_size)+'_delay-30min.html', node_size=36, edge_width=18)
create_network_visualization(synchronous_div, synchronous_div_degrees, 'synchronous_div_filament_size_'+str(min_fil_size)+'_delay_0min.html', node_size=36, edge_width=18)

In [31]:
def create_combined_network_visualization(graph, filename, node_size=38, edge_width=14, min_filament_length=3):
    """
    Create a network visualization showing both motifs and filamentous branches.
    
    Args:
        graph: NetworkX graph object
        filename: Output filename for the visualization
        node_size: Size of nodes in the visualization
        edge_width: Width of edges in the visualization
        min_filament_length: Minimum length for filament detection
    """
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    # Identify motif nodes (nodes with 2 or more degree-1 neighbors)
    motif_nodes = [node for node in graph.nodes() 
                   if sum(1 for neighbor in graph.neighbors(node) 
                         if graph.degree(neighbor) == 1) >= 2]
    
    # Identify degree-1 nodes connected to motif centers
    motif_degree1_nodes = set()
    for motif_node in motif_nodes:
        for neighbor in graph.neighbors(motif_node):
            if graph.degree(neighbor) == 1:
                motif_degree1_nodes.add(neighbor)
    
    # Function to identify filament nodes
    def identify_filament_nodes(graph, min_length=3):
        filament_nodes = set()
        visited = set()
        
        def traverse_filament(node):
            path = [node]
            current = node
            visited.add(current)
            
            while True:
                neighbors = list(graph.neighbors(current))
                unvisited_neighbors = [n for n in neighbors if n not in visited]
                
                if len(unvisited_neighbors) == 1 and graph.degree(unvisited_neighbors[0]) <= 2:
                    next_node = unvisited_neighbors[0]
                    path.append(next_node)
                    visited.add(next_node)
                    current = next_node
                else:
                    break
            
            return path if len(path) >= min_length else []
        
        for node, degree in dict(graph.degree()).items():
            if degree == 1 and node not in visited:
                filament = traverse_filament(node)
                filament_nodes.update(filament)
        
        return filament_nodes
    
    # Identify filament nodes
    filament_nodes = identify_filament_nodes(graph, min_filament_length)
    
    # Remove motif-related nodes from filament nodes to avoid overlap
    filament_nodes = filament_nodes - set(motif_nodes) - motif_degree1_nodes
    
    # Color nodes based on their classification
    for node in nt.nodes:
        node_id = node['id']
        node['size'] = node_size
        node['label'] = ''
        
        if node_id in motif_nodes:
            # Motif center nodes - red
            node['color'] = 'red'
        elif node_id in motif_degree1_nodes:
            # Degree-1 nodes connected to motif centers - orange
            node['color'] = 'orange'
        elif node_id in filament_nodes:
            # Filament nodes - blue
            node['color'] = 'blue'
        else:
            # Other nodes - dark gray
            node['color'] = 'darkgray'
    
    # Color edges based on their classification
    for edge in nt.edges:
        from_node = edge['from']
        to_node = edge['to']
        
        # Check if edge is part of a motif (connects motif center to degree-1 node)
        is_motif_edge = (from_node in motif_nodes and to_node in motif_degree1_nodes) or \
                       (to_node in motif_nodes and from_node in motif_degree1_nodes)
        
        # Check if edge is part of a filament (connects two filament nodes)
        is_filament_edge = from_node in filament_nodes and to_node in filament_nodes
        
        if is_motif_edge:
            edge['color'] = 'red'
            edge['width'] = edge_width + 4  # Slightly thicker for motif edges
        elif is_filament_edge:
            edge['color'] = 'blue'
            edge['width'] = edge_width + 4  # Slightly thicker for filament edges
        else:
            edge['color'] = 'darkgray'
            edge['width'] = edge_width
    
    # Configure visualization settings
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

# Example usage:
# create_combined_network_visualization(graph, 'combined_network.html', node_size=38, edge_width=14, min_filament_length=3)

In [32]:
create_combined_network_visualization(fast_first_div, 'fast_first_combined_viz_delay-30min.html', node_size=36, edge_width=18, min_filament_length=3)
create_combined_network_visualization(synchronous_div, 'synchronous_combined_viz_delay_0min.html', node_size=36, edge_width=18, min_filament_length=3)